#Build a Robust NLP Preprocessing Engine (Advanced)

###Task 1: Conceptual Understanding

### 1. Difference between "Love" and "love" in NLP
In NLP, "Love" and "love" are treated as different tokens because NLP models are case-sensitive.

### 2. What happens if stopwords are not removed?
If stopwords are not removed, the dataset contains many common words like "is", "the", "and", which add noise.

### 3. When removing stopwords is harmful
- Sentiment analysis: "I am not happy" -> removing "not" changes meaning
- Question answering: "What is the best course?" -> removing "what" affects intent

### 4. Stemming vs Lemmatization
Stemming reduces words to root form (running -> run).
Lemmatization converts words to meaningful base form (better -> good).

###TASK 2 — PREPROCESS FUNCTION

In [2]:
import re

def preprocess_text(text):
    if not text or not isinstance(text, str):
        return [], ""

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)

    # Remove emails
    text = re.sub(r"\S+@\S+", "", text)

    # Remove numbers
    text = re.sub(r"\d+", "", text)

    # Remove emojis (basic)
    text = re.sub(r"[^\w\s]", "", text)

    # Handle repeated characters (soooo → soo → so)
    text = re.sub(r"(.)\1{2,}", r"\1", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()

    # Remove short tokens except "no", "not"
    filtered_tokens = [
        word for word in tokens
        if len(word) > 2 or word in ["no", "not"]
    ]

    clean_sentence = " ".join(filtered_tokens)

    return filtered_tokens, clean_sentence

###TASK 3 — STRESS TESTING

In [4]:
test_sentences = [
    "Get 100% FREE access now!!!",
    "I absolutely looooved this product",
    "Worst service ever... 0/10",
    "Call me at 9876543210",
    "This is THE best course!!!",
    "Visit https://openai.com now!",
    "Nooooo this is baaad!!!",
    "OK OK OK I got it",
    "Win $$$ now!!! Limited offer!!!",
    "I am not happy with this"
]

for text in test_sentences:
    tokens, clean = preprocess_text(text)

    print("\nOriginal:", text)
    print("Tokens:", tokens)
    print("Clean:", clean)


Original: Get 100% FREE access now!!!
Tokens: ['get', 'free', 'access', 'now']
Clean: get free access now

Original: I absolutely looooved this product
Tokens: ['absolutely', 'loved', 'this', 'product']
Clean: absolutely loved this product

Original: Worst service ever... 0/10
Tokens: ['worst', 'service', 'ever']
Clean: worst service ever

Original: Call me at 9876543210
Tokens: ['call']
Clean: call

Original: This is THE best course!!!
Tokens: ['this', 'the', 'best', 'course']
Clean: this the best course

Original: Visit https://openai.com now!
Tokens: ['visit', 'now']
Clean: visit now

Original: Nooooo this is baaad!!!
Tokens: ['no', 'this', 'bad']
Clean: no this bad

Original: OK OK OK I got it
Tokens: ['got']
Clean: got

Original: Win $$$ now!!! Limited offer!!!
Tokens: ['win', 'now', 'limited', 'offer']
Clean: win now limited offer

Original: I am not happy with this
Tokens: ['not', 'happy', 'with', 'this']
Clean: not happy with this


###TASK 4 — TOKEN ANALYTICS

In [5]:
import numpy as np

def analyze(tokens):
    total = len(tokens)
    unique = len(set(tokens))
    avg_len = np.mean([len(t) for t in tokens]) if tokens else 0

    return total, unique, round(avg_len, 2)

for text in test_sentences:
    tokens, _ = preprocess_text(text)
    total, unique, avg = analyze(tokens)

    print(f"\n{text}")
    print("Total:", total)
    print("Unique:", unique)
    print("Avg Length:", avg)


Get 100% FREE access now!!!
Total: 4
Unique: 4
Avg Length: 4.0

I absolutely looooved this product
Total: 4
Unique: 4
Avg Length: 6.5

Worst service ever... 0/10
Total: 3
Unique: 3
Avg Length: 5.33

Call me at 9876543210
Total: 1
Unique: 1
Avg Length: 4.0

This is THE best course!!!
Total: 4
Unique: 4
Avg Length: 4.25

Visit https://openai.com now!
Total: 2
Unique: 2
Avg Length: 4.0

Nooooo this is baaad!!!
Total: 3
Unique: 3
Avg Length: 3.0

OK OK OK I got it
Total: 1
Unique: 1
Avg Length: 3.0

Win $$$ now!!! Limited offer!!!
Total: 4
Unique: 4
Avg Length: 4.5

I am not happy with this
Total: 4
Unique: 4
Avg Length: 4.0


###TASK 5 — FREQUENCY ANALYSIS

In [6]:
from collections import Counter

all_tokens = []

for text in test_sentences:
    tokens, _ = preprocess_text(text)
    all_tokens.extend(tokens)

counter = Counter(all_tokens)

print("Top 10 Frequent Words:")
print(counter.most_common(10))

print("\nTop 5 Least Frequent Words:")
print(counter.most_common()[-5:])

Top 10 Frequent Words:
[('this', 4), ('now', 3), ('get', 1), ('free', 1), ('access', 1), ('absolutely', 1), ('loved', 1), ('product', 1), ('worst', 1), ('service', 1)]

Top 5 Least Frequent Words:
[('limited', 1), ('offer', 1), ('not', 1), ('happy', 1), ('with', 1)]


###TASK 6 — FULL PIPELINE

In [7]:
def full_pipeline(text_list):
    all_tokens = []
    clean_sentences = []

    for text in text_list:
        tokens, clean = preprocess_text(text)
        all_tokens.extend(tokens)
        clean_sentences.append(clean)

    return {
        "tokens": all_tokens,
        "clean_sentences": clean_sentences
    }


output = full_pipeline(test_sentences)
print(output)

{'tokens': ['get', 'free', 'access', 'now', 'absolutely', 'loved', 'this', 'product', 'worst', 'service', 'ever', 'call', 'this', 'the', 'best', 'course', 'visit', 'now', 'no', 'this', 'bad', 'got', 'win', 'now', 'limited', 'offer', 'not', 'happy', 'with', 'this'], 'clean_sentences': ['get free access now', 'absolutely loved this product', 'worst service ever', 'call', 'this the best course', 'visit now', 'no this bad', 'got', 'win now limited offer', 'not happy with this']}


###TASK 7 — ERROR HANDLING

In [8]:
edge_cases = ["", "😍", "123456"]

for text in edge_cases:
    tokens, clean = preprocess_text(text)
    print("\nInput:", text)
    print("Tokens:", tokens)
    print("Clean:", clean)


Input: 
Tokens: []
Clean: 

Input: 😍
Tokens: []
Clean: 

Input: 123456
Tokens: []
Clean: 
